# AI2 - Homework 2: Ensemble των 3 μοντέλων (supplementary)

Αυτό το notebook είναι **εκτός των 3 model-specific deliverables**. Είναι ένα
συμπληρωματικό πείραμα για τον leaderboard score: τρέχω fine-tuning και στα 3 required
μοντέλα (DistilBERT, BERT, DeBERTa-v3) με 3 seeds το καθένα (42, 0, 1), και μετά
μέσο όρο των softmax probabilities για να πάρω ensemble predictions.

**Γιατί**: κάθε μοντέλο/seed συγκλίνει σε διαφορετικό local minimum και κάνει ελαφρώς
διαφορετικά λάθη. Ο μέσος όρος 9 probability distributions συνήθως δίνει +0.02-0.05
macro-F1 πάνω από το καλύτερο single model. Είναι classic variance-reduction τεχνική.

**Single-model best (confirmed)**:
- DistilBERT ml=512 seed=42 wd=0.0: val_f1_macro = 0.6405
- BERT ml=256 seed=1 wd=0.0: val_f1_macro = 0.6417
- DeBERTa ml=256 seed=42 wd=0.0: val_f1_macro = 0.6962

Στόχος του ensemble: να ξεπεράσω το 0.6962 στο val set και να βελτιώσω το leaderboard
score.

**Runtime estimate**: ~3-3.5 ώρες σε Kaggle T4 (9 trainings back-to-back).

## Βήμα 1 - Εγκατάσταση dependencies

Pin `transformers==4.44.0` για reproducibility (DeBERTa-v3 crashes με 5.0.0).

In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys

# Pin transformers to 4.44.0 — transformers 5.0.0 has a broken DeBERTa-v3 fine-tuning issue.
# After this install, the kernel MUST be restarted before running Cell 2+.
# On Kaggle: Run this cell alone first → Restart kernel → Run all remaining cells.
result = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.44.0", "datasets", "accelerate", "scikit-learn",
    "sentencepiece", "protobuf"], capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else "")
print(result.stderr[-300:] if result.stderr else "")

import transformers
print(f"transformers version: {transformers.__version__}")
assert transformers.__version__.startswith("4."), \
    f"STOP: transformers is {transformers.__version__}. Restart the kernel and re-run."
print("deps ok")

## Βήμα 2 - Library (inlined src/)

Ολόκληρο το training pipeline σε ένα cell, όπως στα model-specific notebooks.

In [ ]:
# Cell 2 — Library (inlined src/)
# ============================================================
# src/config.py
# ============================================================
from dataclasses import dataclass, field, asdict
from typing import Literal
import json

@dataclass
class ExperimentConfig:
    model_name: str = "distilbert-base-uncased"
    input_fmt: Literal["two_segment", "concat_sep"] = "two_segment"
    max_length: int = 128
    lr: float = 2e-5
    batch_size: int = 32
    epochs: int = 3
    warmup_steps: int = 0
    grad_clip: float = 1.0
    use_class_weights: bool = False
    weight_decay: float = 0.0
    seed: int = 42
    split_id: int = 0
    mode: Literal["smoke", "dev", "confirm", "final"] = "dev"
    smoke_n: int = 50
    data_dir: str = "data"
    models_dir: str = "/kaggle/working/models"
    run_id: str = field(init=False)

    def __post_init__(self):
        short_model = self.model_name.split("/")[-1]
        self.run_id = (
            f"{short_model}_{self.input_fmt}_lr{self.lr}_bs{self.batch_size}"
            f"_ep{self.epochs}_ml{self.max_length}_seed{self.seed}"
            + ("_cw" if self.use_class_weights else "")
            + (f"_wd{self.weight_decay}" if self.weight_decay > 0 else "")
        )

    def to_dict(self) -> dict:
        return asdict(self)

    def save(self, path: str) -> None:
        with open(path, "w") as f:
            json.dump(self.to_dict(), f, indent=2)

# ============================================================
# src/data.py
# ============================================================
from typing import Optional, Tuple
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

LABEL2ID = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = 3
HF_DATASET = "ailsntua/QEvasion"
CORRUPTED_TRAIN_INDEX_VALUES = list(range(1870, 1884))

def load_clarity(data_dir=None):
    ds = load_dataset(HF_DATASET)
    train = ds["train"].to_pandas().copy()
    test = ds["test"].to_pandas().copy()
    for df in (train, test):
        df.rename(columns={"interview_answer": "answer", "clarity_label": "label"}, inplace=True)
    return train, test

def clean_data(df, verbose=True):
    raw_n = len(df)
    qa_cols = ["question", "answer"]
    dup_mask = df.duplicated(subset=qa_cols, keep=False)
    conflicting_ids = (
        df[dup_mask].groupby(qa_cols)["label"].nunique()
        .pipe(lambda s: s[s > 1]).index
    )
    multi_idx = pd.MultiIndex.from_frame(df[qa_cols])
    conflict_multi = pd.MultiIndex.from_tuples(conflicting_ids)
    drop_mask = multi_idx.isin(conflict_multi)
    cleaned = df[~drop_mask].copy()
    after_conflicts = len(cleaned)
    cleaned = cleaned.loc[~cleaned["index"].isin(CORRUPTED_TRAIN_INDEX_VALUES)].reset_index(drop=True)
    after_corrupted = len(cleaned)
    if verbose:
        print(f"[clean_data] {raw_n} -> {after_conflicts} (−{raw_n-after_conflicts} conflicts) -> {after_corrupted} (−{after_conflicts-after_corrupted} corrupted)")
    return cleaned

def encode_labels(df, label_col="label"):
    df = df.copy()
    df["label_id"] = df[label_col].map(LABEL2ID)
    return df

def create_split(df, val_size=0.1, seed=0):
    idx = np.arange(len(df))
    train_idx, val_idx = train_test_split(
        idx, test_size=val_size, random_state=seed, stratify=df["label_id"].values
    )
    return train_idx, val_idx

def smoke_subset(df, n_per_class=50, seed=42):
    parts = [g.sample(min(n_per_class, len(g)), random_state=seed) for _, g in df.groupby("label_id")]
    return pd.concat(parts, ignore_index=True)

def subgroup_metadata(df):
    df = df.copy()
    df["q_len"] = df["question"].str.split().str.len()
    df["a_len"] = df["answer"].str.split().str.len()
    df["q_len_bin"] = pd.qcut(df["q_len"], q=3, labels=["short", "medium", "long"])
    df["a_len_bin"] = pd.qcut(df["a_len"], q=3, labels=["short", "medium", "long"])
    return df

# ============================================================
# src/tokenization.py
# ============================================================
import torch
from torch.utils.data import TensorDataset

def tokenize_pairs(df, tokenizer, max_length, input_fmt="two_segment"):
    questions = df["question"].astype(str).tolist()
    answers = df["answer"].astype(str).tolist()
    if input_fmt == "two_segment":
        enc = tokenizer(questions, answers, padding="max_length", truncation=True,
                        max_length=max_length, return_tensors="pt")
    elif input_fmt == "concat_sep":
        sep = tokenizer.sep_token or "[SEP]"
        texts = [f"{q} {sep} {a}" for q, a in zip(questions, answers)]
        enc = tokenizer(texts, padding="max_length", truncation=True,
                        max_length=max_length, return_tensors="pt")
    else:
        raise ValueError(f"Unknown input_fmt: {input_fmt!r}")
    labels = torch.tensor(df["label_id"].values, dtype=torch.long)
    return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)

# ============================================================
# src/model.py
# ============================================================
from transformers import AutoModelForSequenceClassification, AutoTokenizer

def load_model_and_tokenizer(model_name, num_labels=3):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    return model, tokenizer

# ============================================================
# src/train.py
# ============================================================
import random
from torch.utils.data import DataLoader

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def train_one_epoch(model, loader, optimizer, scheduler, device, grad_clip=1.0, class_weights=None):
    model.train()
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights) if class_weights is not None else None
    total_loss = 0.0
    for batch in loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        optimizer.zero_grad()
        if loss_fn is not None:
            out = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(out.logits, labels)
        else:
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)

# ============================================================
# src/evaluate.py
# ============================================================
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    total_loss = 0.0
    for batch in loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += out.loss.item()
        all_logits.append(out.logits.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    logits = np.concatenate(all_logits, axis=0)
    labels_np = np.concatenate(all_labels, axis=0)
    preds = logits.argmax(axis=1)
    metrics = {
        "val_loss": total_loss / max(len(loader), 1),
        "accuracy": float(accuracy_score(labels_np, preds)),
        "f1_macro": float(f1_score(labels_np, preds, average="macro", zero_division=0)),
        "precision_macro": float(precision_score(labels_np, preds, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(labels_np, preds, average="macro", zero_division=0)),
        "f1_per_class": f1_score(labels_np, preds, average=None, labels=[0,1,2], zero_division=0).tolist(),
    }
    return metrics, logits, labels_np

# ============================================================
# src/experiment.py
# ============================================================
from pathlib import Path
from torch.optim import AdamW
from torch.utils.data import RandomSampler, SequentialSampler
from transformers import get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight

def run_experiment(cfg):
    set_seed(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[info] device: {device}")

    train_df_raw, test_df = load_clarity(cfg.data_dir)
    cleaned = clean_data(train_df_raw)
    cleaned = encode_labels(cleaned)

    if cfg.mode == "smoke":
        cleaned = smoke_subset(cleaned, n_per_class=cfg.smoke_n, seed=cfg.seed)
        print(f"[info] smoke subset: {len(cleaned)} rows")

    train_idx, val_idx = create_split(cleaned, val_size=0.1, seed=cfg.split_id)
    train_df_split = cleaned.iloc[train_idx].reset_index(drop=True)
    val_df_split = cleaned.iloc[val_idx].reset_index(drop=True)
    print(f"[info] train: {len(train_df_split)}, val: {len(val_df_split)}")

    # class weights computed from training split only
    class_weights = None
    if cfg.use_class_weights:
        weights = compute_class_weight(
            "balanced", classes=np.array([0, 1, 2]), y=train_df_split["label_id"].values
        )
        class_weights = torch.tensor(weights, dtype=torch.float).to(device)
        print(f"[info] class weights: CR={weights[0]:.3f}  AMB={weights[1]:.3f}  CNR={weights[2]:.3f}")

    model, tokenizer = load_model_and_tokenizer(cfg.model_name, num_labels=NUM_LABELS)
    model.to(device)

    train_ds = tokenize_pairs(train_df_split, tokenizer, cfg.max_length, cfg.input_fmt)
    val_ds = tokenize_pairs(val_df_split, tokenizer, cfg.max_length, cfg.input_fmt)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=RandomSampler(train_ds))
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, sampler=SequentialSampler(val_ds))

    if "deberta" in cfg.model_name.lower():
        head_params = list(model.pooler.parameters()) + list(model.classifier.parameters())
        head_ids = {id(p) for p in head_params}
        backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
        optimizer = AdamW([
            {"params": backbone_params, "lr": cfg.lr},
            {"params": head_params,     "lr": cfg.lr * 10},
        ], eps=1e-6, weight_decay=cfg.weight_decay)
    else:
        optimizer = AdamW(model.parameters(), lr=cfg.lr, eps=1e-6, weight_decay=cfg.weight_decay)

    total_steps = len(train_loader) * cfg.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=cfg.warmup_steps, num_training_steps=total_steps
    )

    run_dir = Path(cfg.models_dir) / cfg.run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    history = []
    val_logits = np.array([])
    val_labels_arr = np.array([])
    best_f1 = -1.0
    for epoch in range(cfg.epochs):
        train_loss = train_one_epoch(
            model, train_loader, optimizer, scheduler, device, cfg.grad_clip, class_weights
        )
        val_metrics, val_logits, val_labels_arr = evaluate(model, val_loader, device)
        print(
            f"[epoch {epoch+1}/{cfg.epochs}] "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['val_loss']:.4f} "
            f"val_f1_macro={val_metrics['f1_macro']:.4f} "
            f"val_acc={val_metrics['accuracy']:.4f}"
        )
        history.append({"epoch": epoch+1, "train_loss": train_loss, **val_metrics})
        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            torch.save(model.state_dict(), run_dir / "best_model.pt")
            np.save(run_dir / "val_preds.npy", val_logits)
            np.save(run_dir / "val_labels.npy", val_labels_arr)
            print(f"  * new best checkpoint (f1_macro={best_f1:.4f})")

    model.load_state_dict(torch.load(run_dir / "best_model.pt", map_location=device))
    val_logits = np.load(run_dir / "val_preds.npy")
    val_labels_arr = np.load(run_dir / "val_labels.npy")

    cfg.save(str(run_dir / "config.json"))
    with open(run_dir / "metrics.json", "w") as f:
        json.dump(history, f, indent=2)
    print(f"[info] saved run to {run_dir} (best f1_macro={best_f1:.4f})")

    return history, val_logits, val_labels_arr, test_df, tokenizer, model, device

print("library loaded ok")

## Βήμα 3 - Configs για τα 9 training runs

Χρησιμοποιώ το best config κάθε μοντέλου (από τα single-model notebooks) και τρέχω 3
seeds: 42, 0, 1. Όλα με explicit `weight_decay=0.0` (βλ. το DistilBERT notebook για το
rationale - το PyTorch AdamW default είναι 0.01 οπότε πρέπει να το θέσω explicit).

In [ ]:
# 9 configs: 3 architectures x 3 seeds, all with wd=0.0, mode='final'
ensemble_configs = []

# DistilBERT: ml=512 gave the best single-seed F1
for seed in [42, 0, 1]:
    ensemble_configs.append(ExperimentConfig(
        model_name="distilbert-base-uncased",
        input_fmt="two_segment",
        max_length=512,
        lr=3e-5,
        batch_size=16,
        epochs=3,
        warmup_steps=100,
        mode="final",
        seed=seed,
        weight_decay=0.0,
    ))

# BERT: ml=256, ep=5
for seed in [42, 0, 1]:
    ensemble_configs.append(ExperimentConfig(
        model_name="bert-base-uncased",
        input_fmt="two_segment",
        max_length=256,
        lr=2e-5,
        batch_size=16,
        epochs=5,
        warmup_steps=100,
        mode="final",
        seed=seed,
        weight_decay=0.0,
    ))

# DeBERTa-v3: ml=256, ep=3
for seed in [42, 0, 1]:
    ensemble_configs.append(ExperimentConfig(
        model_name="microsoft/deberta-v3-base",
        input_fmt="two_segment",
        max_length=256,
        lr=2e-5,
        batch_size=16,
        epochs=3,
        warmup_steps=100,
        mode="final",
        seed=seed,
        weight_decay=0.0,
    ))

print(f"Total configs: {len(ensemble_configs)}")
for i, cfg in enumerate(ensemble_configs):
    print(f"  [{i+1}] {cfg.run_id}")

## Βήμα 4 - Training loop + test inference

Για κάθε config:
1. Τρέχω `run_experiment` (κάνει fine-tuning, σώζει το best checkpoint με βάση val F1).
2. Φορτώνω ξανά το best checkpoint, προβλέπω softmax probs πάνω στο **val** και στο **test**.
3. Αποθηκεύω τα probs για το ensemble.

Το val split είναι fixed (`split_id=0`, stratified), άρα τα val labels είναι ίδια
για όλα τα 9 runs - μπορώ να υπολογίσω ensemble val F1 συνεκτικά.

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, SequentialSampler
from scipy.special import softmax

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

# pre-load test data once (load_clarity returns train, test)
_, test_df_raw = load_clarity()
test_df_enc = test_df_raw.copy()
test_df_enc["label_id"] = 0  # dummy labels for tokenizer pipeline

individual_val_f1s = []
test_probs_per_model = []   # each entry: [N_test, 3] softmax array
val_probs_per_model = []    # each entry: [N_val,  3] softmax array
val_labels_ref = None       # same for all runs (fixed split)

for i, cfg in enumerate(ensemble_configs):
    print(f"\n{'='*70}")
    print(f"[{i+1}/{len(ensemble_configs)}] {cfg.run_id}")
    print('='*70)

    # train (saves best_model.pt to models/<run_id>/)
    history = run_experiment(cfg)

    best_epoch = max(history, key=lambda h: h["f1_macro"])
    individual_val_f1s.append(best_epoch["f1_macro"])
    print(f"  best val_f1_macro: {best_epoch['f1_macro']:.4f}")

    # reload best checkpoint + tokenizer for inference
    run_dir = Path(cfg.models_dir) / cfg.run_id
    reload_model, reload_tok = load_model_and_tokenizer(cfg.model_name, num_labels=NUM_LABELS)
    reload_model.load_state_dict(torch.load(run_dir / "best_model.pt", map_location=device))
    reload_model.to(device).eval()

    # test softmax probs
    test_ds = tokenize_pairs(test_df_enc, reload_tok, cfg.max_length, cfg.input_fmt)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, sampler=SequentialSampler(test_ds))

    test_probs_chunks = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids, attention_mask, _ = [b.to(device) for b in batch]
            out = reload_model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
            test_probs_chunks.append(probs)
    test_probs = np.concatenate(test_probs_chunks, axis=0)
    test_probs_per_model.append(test_probs)

    # val softmax probs (reuse saved val_logits, convert to probs)
    val_logits = np.load(run_dir / "val_preds.npy")
    val_labels = np.load(run_dir / "val_labels.npy")
    val_probs = softmax(val_logits, axis=-1)
    val_probs_per_model.append(val_probs)
    if val_labels_ref is None:
        val_labels_ref = val_labels
    else:
        # sanity: fixed split means identical val labels every time
        assert np.array_equal(val_labels_ref, val_labels), "val split drift!"

    # free GPU memory before next model
    del reload_model, reload_tok
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\nAll 9 trainings done.")
print(f"Individual val F1s: {[f'{x:.4f}' for x in individual_val_f1s]}")
print(f"Mean individual F1: {np.mean(individual_val_f1s):.4f}")

## Βήμα 5 - Ensemble computation

Μέσος όρος των 9 softmax probability matrices (stacked κατά μοντέλο), argmax για τις
τελικές προβλέψεις. Υπολογίζω και το val F1 του ensemble για comparison με τα
individual μοντέλα.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report

# stack and average
test_probs_stack = np.stack(test_probs_per_model, axis=0)  # [9, N_test, 3]
val_probs_stack  = np.stack(val_probs_per_model,  axis=0)  # [9, N_val,  3]

ens_test_probs = test_probs_stack.mean(axis=0)
ens_val_probs  = val_probs_stack.mean(axis=0)

ens_test_preds = ens_test_probs.argmax(axis=1)
ens_val_preds  = ens_val_probs.argmax(axis=1)

# val metrics for the ensemble
ens_val_f1_macro = f1_score(val_labels_ref, ens_val_preds, average="macro")
ens_val_acc      = accuracy_score(val_labels_ref, ens_val_preds)

print(f"Ensemble val F1 (macro): {ens_val_f1_macro:.4f}")
print(f"Ensemble val accuracy:   {ens_val_acc:.4f}")
print(f"Best single val F1:      {max(individual_val_f1s):.4f}")
print(f"Improvement over best single: +{ens_val_f1_macro - max(individual_val_f1s):+.4f}")
print()

LABEL_ORDER = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
print(classification_report(val_labels_ref, ens_val_preds, target_names=LABEL_ORDER, digits=3))

## Βήμα 6 - Submission CSV

Κρατάω τις ensemble προβλέψεις στη μορφή `Id, Predicted` που απαιτεί ο διαγωνισμός.

In [ ]:
import pandas as pd

submission = pd.DataFrame({
    "Id": test_df_raw["index"] if "index" in test_df_raw.columns else range(len(test_df_raw)),
    "Predicted": [ID2LABEL[p] for p in ens_test_preds],
})

submission.to_csv("/kaggle/working/submission_ensemble.csv", index=False)
submission.to_csv("/kaggle/working/submission.csv", index=False)
print(f"Wrote submission_ensemble.csv and submission.csv ({len(submission)} rows)")
display(submission.head())
print(submission["Predicted"].value_counts())

## Βήμα 7 - Ανά-μοντέλο diagnostics

Individual val F1 ανά run, για να δω αν κάποιο μοντέλο υστερεί σημαντικά και ενδεχομένως
να το αποκλείσω από το ensemble σε μελλοντική επανάληψη.

In [ ]:
diag_rows = []
for cfg, f1 in zip(ensemble_configs, individual_val_f1s):
    diag_rows.append({
        "run_id": cfg.run_id,
        "model": cfg.model_name.split("/")[-1],
        "seed": cfg.seed,
        "val_f1_macro": round(f1, 4),
    })

diag_df = pd.DataFrame(diag_rows)
display(diag_df)
print(f"\nEnsemble val F1: {ens_val_f1_macro:.4f}")
print(f"Best individual:  {diag_df['val_f1_macro'].max():.4f}  ({diag_df.loc[diag_df['val_f1_macro'].idxmax(), 'run_id']})")
print(f"Worst individual: {diag_df['val_f1_macro'].min():.4f}  ({diag_df.loc[diag_df['val_f1_macro'].idxmin(), 'run_id']})")

## Βήμα 8 - Download submission

Link για να κατεβάσω τοπικά το submission.csv.

In [ ]:
from IPython.display import FileLink
display(FileLink("/kaggle/working/submission_ensemble.csv"))
display(FileLink("/kaggle/working/submission.csv"))